In [1]:
def test_eval_rpn(solution):
    # Test case 1
    tokens = ["2", "1", "+", "3", "*"]
    expected = 9
    print(f"test case: tokens {tokens}")
    result = solution.evalRPN(tokens)
    assert result == expected, f"Test1 failed: got {result}, expected {expected}"

    # Test case 2
    tokens = ["4", "13", "5", "/", "+"]
    expected = 6
    print(f"test case: tokens {tokens}")
    result = solution.evalRPN(tokens)
    assert result == expected, f"Test2 failed: got {result}, expected {expected}"

    # Test case 3
    tokens = ["10", "6", "9", "3", "+", "-11", "*", "/", "*", "17", "+", "5", "+"]
    expected = 22
    print(f"test case: tokens {tokens}")
    result = solution.evalRPN(tokens)
    assert result == expected, f"Test3 failed: got {result}, expected {expected}"

    print("All test cases passed.")


In [6]:
from typing import List

class Solution:
    def eval_operator(self, first, second, operator) -> int:
        if operator == '+':
            return first + second
        elif operator == '*':
            return first * second
        elif operator == '-':
            return first - second
        elif operator == '/':
            return int(first / second)

    def evalRPN(self, tokens: List[str]) -> int:
        stack = []
        res = None
        for i in range(len(tokens)):
            if tokens[i] in ['+', '-', '*', '/']:
                print(f"i: {i}, tokens[{i}]: {tokens[i]}, stack: {stack}")
                first = stack.pop(-1)
                if res == None:
                    second = stack.pop(-1)
                    res = self.eval_operator(int(first), int(second), tokens[i])
                else:
                    res = self.eval_operator(int(res), int(first), tokens[i])
            else:  # number case
                stack.append(int(tokens[i]))
        return res


In [7]:
solution = Solution()
test_eval_rpn(solution)


test case: tokens ['2', '1', '+', '3', '*']
i: 2, tokens[2]: +, stack: [2, 1]
i: 4, tokens[4]: *, stack: [3]
test case: tokens ['4', '13', '5', '/', '+']
i: 3, tokens[3]: /, stack: [4, 13, 5]
i: 4, tokens[4]: +, stack: [4]


AssertionError: Test2 failed: got 4, expected 6

In [8]:
class Solution:
    def eval_operator(self, first, second, operator) -> int:
        if operator == '+':
            return first + second
        elif operator == '*':
            return first * second
        elif operator == '-':
            return first - second
        elif operator == '/':
            return int(first / second)
    def evalRPN(self, tokens: List[str]) -> int:
        stack = []
        for i in range(len(tokens)):
            if tokens[i] in ['+', '-', '*', '/']:
                second = stack.pop()
                first = stack.pop()
                res = self.eval_operator(first, second, tokens[i])
                stack.append(res)
            else: #number case
                stack.append(int(tokens[i]))
        return stack[0]

In [9]:
solution = Solution()
test_eval_rpn(solution)

test case: tokens ['2', '1', '+', '3', '*']
test case: tokens ['4', '13', '5', '/', '+']
test case: tokens ['10', '6', '9', '3', '+', '-11', '*', '/', '*', '17', '+', '5', '+']
test case: tokens ['18']
test case: tokens ['5', '3', '-']
All test cases passed.


# ['10', '6', '9', '3', '+', '-11', '*', '/', '*', '17', '+', '5', '+']
# ((10 * (6 / ((9 + 3) * -11))) + 17 + 5)

## 1. Complexity and Trade-offs of all solution attempts, with the main emphasis on the last attempt.

- This notebook contains two solution attempts. Both target the correct high-level design: evaluate Reverse Polish Notation with a stack.
- The first attempt aimed for `O(n)` time and `O(n)` space, but it was not correct because it split state across `stack` and `res`. That broke the main invariant of the problem: every intermediate result must go back onto the stack so later operators can consume it in the right order.
- The first attempt also handled operand order incorrectly for non-commutative operators. In RPN, if you pop `second` and then `first`, the operation must be `first op second`. That matters for `-` and `/`.
- The final solution keeps the same optimal asymptotic complexity, `O(n)` time and `O(n)` space, but fixes the representation. It uses the stack as the single source of truth, which is the correct trade-off for this problem.
- The final cell is correct for the LeetCode contract. It pops `second`, then `first`, computes the result with the proper operand order, pushes the intermediate value back onto the stack, and returns the remaining value.
- One minor readability note: `return stack[0]` works because valid input leaves exactly one value, but `return stack[-1]` communicates the stack-machine model more clearly. That is a small style issue, not a correctness issue.

## 2. Critique of the problem-solving approach, including progression of thought and method.

- Your progression of thought is strong in the important way: you recognized early that this is a stack-evaluation problem, which is the central insight.
- The first attempt failed because the implementation drifted away from that insight. By introducing `res`, you created a second execution path for intermediate values. That made the stack no longer represent the full execution state.
- The final solution fixed that by restoring the correct invariant: after processing each token, the stack contains exactly the intermediate values needed to evaluate the remaining suffix of the expression.
- Concretely, your final solution fixed the earlier version in three important ways:
- It removed the separate `res` accumulator, so all state now lives in one structure.
- It popped operands in the correct semantic order, `second` then `first`, and evaluated `first op second`.
- It pushed every computed result back onto the stack, which allows chained expressions like Example 3 to work correctly.
- This is a good example of moving from a partially right idea to a clean final implementation by tightening the invariant instead of adding more conditions.

## 3. Improvements to Algorithm/ Optimal Example (include python solution code here in ``` ``` grouping braces)

- Your final algorithm is already the optimal interview approach. The main improvements left are code clarity and a slightly cleaner control flow.
- In modern engineering style, I would simplify the token loop and make the operand roles explicit without indexing into the list.

```python
from typing import List

class Solution:
    def eval_operator(self, first: int, second: int, operator: str) -> int:
        if operator == "+":
            return first + second
        if operator == "-":
            return first - second
        if operator == "*":
            return first * second
        return int(first / second)

    def evalRPN(self, tokens: List[str]) -> int:
        stack = []

        for token in tokens:
            if token in {"+", "-", "*", "/"}:
                second = stack.pop()
                first = stack.pop()
                stack.append(self.eval_operator(first, second, token))
            else:
                stack.append(int(token))

        return stack[-1]
```

- This is functionally the same design as your final solution. The difference is mainly that the invariant is more obvious when iterating directly over `token` and returning `stack[-1]`.

## 4. Applications in real-life situations, with examples from big tech and startups (frontier tech) of the exact problem I'm solving and the solution approach. Give exact examples and usages of the generalization of the solution or pattern. (Use search for examples if needed). Be critical and outline other algorithmic tradeoffs and when I'll use this algorithmic design/ data structure approach and when I should not.

- The exact interview problem is evaluating postfix expressions. The transferable engineering pattern is **stack-based execution of a linear instruction stream where each operation consumes recent outputs and emits a new value back into the same execution state**.
- Literal usage is strongest in interpreters, bytecode runtimes, expression engines, and rule evaluators. The direct transfer is not "companies store user formulas as LeetCode token arrays"; the real transfer is the execution model.
- In big-tech-style systems, this pattern maps to internal execution engines for query filters, policy checks, calculator-like formula evaluation, and compact virtual-machine layers. In startup systems, the same pattern appears in pricing-rule evaluators, workflow-condition engines, feature-flag targeting, and no-code automation runtimes.
- The reason your final solution is closer to production-worthy thinking than the first attempt is that execution engines need one authoritative state model. Your final version has that. The first version did not.
- You should use this design when the expression is already in postfix form, when evaluation is single-pass, and when LIFO operand consumption is the natural model.
- You should not use this design when you need rich parsing diagnostics, algebraic optimization, repeated re-planning, or semantic rewrites. In those cases, an AST or a more explicit intermediate representation is usually better.
- The broader lesson is exact and transferable: the winning idea was not just "use a stack," but "make the stack the entire execution state." That is the specific design improvement your final solution achieved.

### RPN Automaton

Formally, this is **not** a finite automaton problem. A DFA or NFA cannot model RPN evaluation because the machine needs unbounded stack memory. The right automata-theory object is a **deterministic pushdown transducer**: a pushdown automaton whose stack stores semantic values and whose computation returns the final value.

A compact formal model is:

- `M = (Q, Σ, Γ, δ, q0, Z0, F)`
- `Q = {q_read, q_accept, q_error}`
- `Σ = IntTok U {+, -, *, /}` where `IntTok` is the set of integer tokens
- `Γ = Z U {⊥}` where stack symbols are integer values plus the bottom marker `⊥`
- `q0 = q_read`
- `Z0 = ⊥`
- `F = {q_accept}`

Transition semantics:

- If the next input token is an integer `n`, then `δ(q_read, n, S) = (q_read, S · n)`.
- If the next input token is an operator `op` and the stack top has at least two values `... , first, second`, then `δ(q_read, op, S · first · second) = (q_read, S · (first op second))`.
- If the next token is an operator and fewer than two values are available, transition to `q_error`.
- On end of input, transition to `q_accept` iff the stack is exactly `⊥ · v` for some integer `v`; otherwise transition to `q_error`.

The key invariant follows directly from that definition: after consuming any prefix of the token stream, the stack contains exactly the semantic values of the partially evaluated postfix expression for that prefix.

```mermaid
flowchart TD
    start([start]) --> read[q_read]
    read -->|integer n: push n| read
    read -->|operator op and at least 2 values: pop second, pop first, push first op second| read
    read -->|EOF and stack = bottom + one value| accept[q_accept]
    read -->|operator and stack height < 2| error[q_error]
    read -->|EOF and stack is not bottom + one value| error
```

Why your final solution matches the automaton:

- Reading a number token implements the `push(n)` transition.
- Reading an operator token implements the `pop, pop, compute, push` transition.
- Returning the last stack value corresponds to accepting only when exactly one semantic value remains.
- Your earlier `res` version broke this formal model because part of the semantic state lived outside the pushdown store, so the stack no longer represented the machine configuration by itself.
